# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
import gradio as gr
import requests
from openai import OpenAI



In [ ]:
MODEL_GPT    = "gpt-4o-mini"
MODEL_LLAMA  = "llama3.2"
MODEL_CLAUDE = "openai/claude-3-5-sonnet"

In [ ]:
# Load environment variables in a file called .env

openai_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

# Ollama (local)
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# OpenRouter (multi-model gateway — gives access to Claude, Gemini, etc.)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

In [ ]:
SYSTEM_PROMPT = """You are an expert Python and AI engineer with 15 years of experience.
You explain technical concepts with extreme clarity — always starting with a one-sentence
plain-English summary, then going deeper with code examples when helpful.
You anticipate follow-up confusion and address it proactively.
Keep answers focused and practical."""


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_stackoverflow",
            "description": "Search Stack Overflow for technical questions. Use this when the user asks something that would benefit from community-validated answers or code examples.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query, e.g. 'python yield from generator'"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

def search_stackoverflow(query: str) -> str:
    """Call the Stack Overflow API and return top result titles + links."""
    url = "https://api.stackexchange.com/2.3/search/advanced"
    params = {
        "order": "desc",
        "sort": "relevance",
        "q": query,
        "site": "stackoverflow",
        "pagesize": 3,
        "filter": "withbody"
    }
    try:
        resp = requests.get(url, params=params, timeout=5)
        data = resp.json()
        items = data.get("items", [])
        if not items:
            return "No results found on Stack Overflow."
        results = []
        for item in items:
            title = item.get("title", "No title")
            link  = item.get("link", "")
            score = item.get("score", 0)
            results.append(f"• [{title}]({link})  (score: {score})")
        return "\n".join(results)
    except Exception as e:
        return f"Stack Overflow search failed: {e}"

def handle_tool_call(tool_name: str, tool_args: dict) -> str:
    """Route tool calls to the right function."""
    if tool_name == "search_stackoverflow":
        return search_stackoverflow(tool_args["query"])
    return "Unknown tool."

In [ ]:
def get_client_and_model(model_choice: str):
    """Return the right (client, model_string) pair based on dropdown selection."""
    if model_choice == "GPT-4o-mini":
        return openai_client, MODEL_GPT
    elif model_choice == "Llama 3.2 (local)":
        return ollama_client, MODEL_LLAMA
    elif model_choice == "Claude 3.5 Sonnet (OpenRouter)":
        return openrouter_client, MODEL_CLAUDE
    else:
        return openai_client, MODEL_GPT


def chat(user_message: str, history: list, model_choice: str):
    """
    Main chat function called by Gradio on every user message.
    - history: list of {"role": ..., "content": ...} dicts (Gradio manages this)
    - Streams tokens back to the UI as they arrive
    - Handles tool calls transparently
    """
    client, model = get_client_and_model(model_choice)

    # Build the full messages list: system + history + new user message
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history
    messages.append({"role": "user", "content": user_message})

    # ── First API call (may trigger a tool) ──────────────────────────────────
    # Note: streaming + tools requires a small workaround — we do a non-streaming
    # first pass to detect tool calls, then stream the final answer.
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # ── If the model wants to call a tool ────────────────────────────────────
    if message.tool_calls:
        tool_call    = message.tool_calls[0]
        tool_name    = tool_call.function.name
        tool_args    = eval(tool_call.function.arguments)   # safe here; it's our own schema
        tool_result  = handle_tool_call(tool_name, tool_args)

        # Feed the tool result back and stream the final response
        messages.append(message)   # assistant's tool-call request
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result
        })

    # ── Stream the final answer ───────────────────────────────────────────────
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    partial = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            partial += delta
            yield partial  

In [ ]:

with gr.Blocks(title="Technical Q&A Assistant", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🧠 Technical Q&A Assistant
    *Your expert Python & AI tutor — powered by multiple LLMs*
    """)

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=[
                "GPT-4o-mini",
                "Claude 3.5 Sonnet (OpenRouter)",
                "Llama 3.2 (local)",
            ],
            value="GPT-4o-mini",
            label="🤖 Choose Model",
            scale=1
        )

    chatbot = gr.ChatInterface(
        fn=chat,
        additional_inputs=[model_dropdown],
        type="messages",
        chatbot=gr.Chatbot(
            height=500,
            type="messages",        # must match ChatInterface type
            render_markdown=True,   # renders code blocks, bold, etc.
            label="Assistant"
        ),
        textbox=gr.Textbox(
            placeholder="Ask a technical question... e.g. 'What does yield from do in Python?'",
            container=False,
            scale=7
        ),
        examples=[
            # Each inner list = [message, model_choice]
            ["What does `yield from` do in Python?",               "GPT-4o-mini"],
            ["Explain the difference between a thread and a process.", "GPT-4o-mini"],
            ["What is RAG and why is it useful?",                   "Claude 3.5 Sonnet (OpenRouter)"],
            ["Search Stack Overflow for: python async vs threading", "GPT-4o-mini"],
        ],
        title=None,   # we already have our own title above
    )

    gr.Markdown("""
    ---
    💡 **Tip:** Mention "search Stack Overflow" in your question to trigger the tool!
    """)

demo.launch()


